# 05 Place Catalog Validation

## 목적
서울/한양 관광 도슨트 RAG에서 사용할 place catalog seed의 정량 gate와 정성 리스크를 확인한다.

이 notebook은 retrieval 성능 평가가 아니다. query rewrite와 place-aware retrieval이 참조할 공개 가능한 장소 기준 데이터를 검증한다.


In [ ]:
from pathlib import Path

from app.domain.place_catalog import (
    build_place_catalog_report,
    collect_place_catalog_gate_failures,
    load_place_catalog,
)

catalog_path = Path('../data_samples/place_catalog_seed.json')
catalog = load_place_catalog(catalog_path)
report = build_place_catalog_report(catalog)
catalog_path


## 정량 Gate

Hard gate는 duplicate, unknown relation, public leakage가 모두 0이어야 한다.


In [ ]:
summary = report.quality_summary
{
    'place_count': summary.place_count,
    'alias_count': summary.alias_count,
    'relation_count': summary.relation_count,
    'minimum_place_count': summary.minimum_place_count,
    'minimum_alias_count': summary.minimum_alias_count,
    'duplicate_place_id_count': summary.duplicate_place_id_count,
    'duplicate_canonical_name_count': summary.duplicate_canonical_name_count,
    'duplicate_alias_count': summary.duplicate_alias_count,
    'unknown_related_place_count': summary.unknown_related_place_count,
    'self_relation_count': summary.self_relation_count,
    'public_raw_text_leakage_count': summary.public_raw_text_leakage_count,
    'private_path_leakage_count': summary.private_path_leakage_count,
    'secret_like_leakage_count': summary.secret_like_leakage_count,
}


In [ ]:
collect_place_catalog_gate_failures(report)


## 분포 확인

장소 유형, alias 언어, relation type 분포를 확인한다. 이 분포는 retrieval 성능 주장이 아니라 다음 실험의 coverage 점검이다.


In [ ]:
report.place_count_by_category


In [ ]:
report.alias_count_by_language


In [ ]:
report.relation_count_by_type


In [ ]:
report.qualitative_assessment


## 판정

현재 place catalog gate는 통과했다. 다음 단계는 이 seed를 바로 generation에 넣는 것이 아니라 Dense/Hybrid retrieval 비교와 place-aware query rewrite ablation의 기준 dimension으로 사용하는 것이다.
